# Tutorial 04: Regularized Coulomb Potential in 1D and 2D

> **Language / Idioma:** [Español](../es/04_potencial_coulomb.ipynb) | **English**

This notebook explores relativistic wavepacket dynamics in a regularized Coulomb potential:
$$V(\mathbf{r}) = -\frac{Z}{\|\mathbf{r}\| + \epsilon}$$
where $Z$ is the effective nuclear charge and $\epsilon > 0$ regularizes the $r \to 0$ singularity.

In [ ]:
import sys
sys.path.append('../..')
import numpy as np
import matplotlib.pyplot as plt
from dirac_solver import (
    Grid, ConstantSpinor, GaussianPacket,
    DiracProblemBuilder, DiracSolver, CoulombPotential, Observables, _core
)

## 1. 2D Scattering on a Regularized Coulomb Center

In [ ]:
grid = Grid(shape=(100, 100), spacing=(0.1, 0.1))
spinor = ConstantSpinor(momentum=[3.0, 0.5, 0.0], mass=1.0)
packet = GaussianPacket(spinor, center=[-2.5, -0.8, 0.0], spatial_width=1.0)

potential = CoulombPotential(Z=0.8, epsilon=0.1)
boundary = _core.AbsorbingBoundary(strength=0.15, width=12)

problem = (
    DiracProblemBuilder()
    .set_grid(grid)
    .set_initial_state(packet)
    .set_potential(potential)
    .set_boundary_condition(boundary)
    .set_time_parameters(time_step=0.002, total_time=4.0)
    .build()
)

solver = DiracSolver(problem)
obs = Observables(solver)
solver.run_simulation(observables_handler=obs)
solver.plot_probability_density()

## 2. Trajectory of Expectation Values $\langle \mathbf{r} \rangle$, $\langle \mathbf{p} \rangle$, and $\langle H \rangle$

In [ ]:
times = obs.get_history('time')
pos = obs.get_history('expected_position')
mom = obs.get_history('expected_momentum')
energy = obs.get_history('expected_energy')

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
axes[0].plot(times, pos[:, 0], label='$\\langle x \\rangle$')
axes[0].plot(times, pos[:, 1], label='$\\langle y \\rangle$')
axes[0].set_ylabel('Position')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(times, mom[:, 0], label='$\\langle p_x \\rangle$')
axes[1].plot(times, mom[:, 1], label='$\\langle p_y \\rangle$')
axes[1].set_ylabel('Momentum')
axes[1].legend(); axes[1].grid(True)

axes[2].plot(times, energy, color='g')
axes[2].set_ylabel('Energy $\\langle H \\rangle$')
axes[2].set_xlabel('Time $t$')
axes[2].grid(True)
plt.tight_layout()
plt.show()